# Performance Analysis

This notebook analyzes the performance metrics collected from Argus, Zizmor, and Ades benchmarking runs.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

df_argus = pd.read_csv('results/benchmark_results_argus.csv')
df_argus['tool'] = 'argus'
print(f"Loaded {len(df_argus)} Argus benchmark results")

df_zizmor = pd.read_csv('results/benchmark_results_zizmor.csv')
df_zizmor['tool'] = 'zizmor'
print(f"Loaded {len(df_zizmor)} zizmor benchmark results")

df_ades = pd.read_csv('results/benchmark_results_ades.csv')
df_ades['tool'] = 'ades'
print(f"Loaded {len(df_ades)} Ades benchmark results")

df = pd.concat([df_argus, df_zizmor, df_ades], ignore_index=True)
print(f"Loaded {len(df)} total benchmark results")

baseline_path = 'results/baseline_results.csv'
if os.path.exists(baseline_path):
    baseline = pd.read_csv(baseline_path).iloc[0]
    print(f"Loaded baseline statistics (collected {baseline['timestamp']})")
else:
    baseline = None
    print("No baseline_results.csv found - baseline overlays will be skipped")

df.head()

## 0. Baseline Statistics

Idle system resource usage collected before any benchmark runs (10-second dstat sample).

In [ ]:
if baseline is not None:
    bl_display = pd.DataFrame([{
        'avg_cpu_percent':   baseline['avg_cpu_percent'],
        'peak_memory_mb':    baseline['peak_memory_mb'],
        'avg_disk_read_kb':  baseline['avg_disk_read_kb'],
        'avg_disk_write_kb': baseline['avg_disk_write_kb'],
        'avg_net_recv_kb':   baseline['avg_net_recv_kb'],
        'avg_net_send_kb':   baseline['avg_net_send_kb'],
    }], index=['baseline'])
    print("Baseline (idle) resource usage:")
    display(bl_display)
else:
    print("Baseline data not available.")


## 1. Summary Statistics

In [ ]:
print("Overall Statistics:")
print(df[['execution_time_seconds', 'avg_cpu_percent', 'peak_memory_mb']].describe())

print("\nPer-Workflow Averages:")
print(df.groupby('workflow_file')[['execution_time_seconds', 'avg_cpu_percent', 'peak_memory_mb']].mean())

## 2. Execution Time Analysis

In [ ]:
# Box plot of execution times
plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x='workflow_file', y='execution_time_seconds')
plt.title('Execution Time Distribution by Workflow')
plt.xlabel('Workflow')
plt.ylabel('Execution Time (seconds)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Variance analysis
print("\nExecution Time Variance:")
variance = df.groupby('workflow_file')['execution_time_seconds'].agg(['mean', 'std', 'min', 'max'])
variance['cv'] = (variance['std'] / variance['mean']) * 100  # Coefficient of variation
print(variance)

## 3. Resource Usage Analysis

In [ ]:
# CPU and Memory usage
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# CPU usage
sns.barplot(data=df, x='workflow_file', y='avg_cpu_percent', ax=axes[0], errorbar=None)
axes[0].set_title('Average CPU Usage by Workflow')
axes[0].set_xlabel('Workflow')
axes[0].set_ylabel('CPU Usage (%)')
axes[0].tick_params(axis='x', rotation=45)
#if baseline is not None:
#    axes[0].axhline(baseline['avg_cpu_percent'], color='red', linestyle='--', linewidth=1.5, label=f"Baseline ({baseline['avg_cpu_percent']:.1f}%)")
#    axes[0].legend()

# Memory usage
sns.barplot(data=df, x='workflow_file', y='peak_memory_mb', ax=axes[1], errorbar=None)
axes[1].set_title('Peak Memory Usage by Workflow')
axes[1].set_xlabel('Workflow')
axes[1].set_ylabel('Memory (MB)')
axes[1].tick_params(axis='x', rotation=45)
#if baseline is not None:
#    axes[1].axhline(baseline['peak_memory_mb'], color='red', linestyle='--', linewidth=1.5, label=f"Baseline ({baseline['peak_memory_mb']:.1f} MB)")
#    axes[1].legend()

plt.tight_layout()
plt.savefig('results/perf_cpu_memory.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Disk and Network I/O

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 10))

# Disk usage
disk_data = df.groupby('workflow_file')[['avg_disk_read_kb', 'avg_disk_write_kb']].mean()
disk_data.plot(kind='bar', ax=axes[0])
axes[0].set_title('Average Disk I/O by Workflow')
axes[0].set_ylabel('KB/s')
axes[0].set_xlabel('Workflow')
axes[0].legend(['Read', 'Write'])
axes[0].tick_params(axis='x', rotation=45)
#if baseline is not None:
#    axes[0].axhline(baseline['avg_disk_read_kb'],  color='tab:blue',   linestyle='--', linewidth=1.5, label=f"Baseline Read ({baseline['avg_disk_read_kb']:.1f} KB/s)")
#    axes[0].axhline(baseline['avg_disk_write_kb'], color='tab:orange', linestyle='--', linewidth=1.5, label=f"Baseline Write ({baseline['avg_disk_write_kb']:.1f} KB/s)")
#    axes[0].legend()

# Network usage
net_data = df.groupby('workflow_file')[['avg_net_recv_kb', 'avg_net_send_kb']].mean()
net_data.plot(kind='bar', ax=axes[1])
axes[1].set_title('Average Network I/O by Workflow')
axes[1].set_ylabel('KB/s')
axes[1].set_xlabel('Workflow')
axes[1].legend(['Receive', 'Send'])
axes[1].tick_params(axis='x', rotation=45)
#if baseline is not None:
#    axes[1].axhline(baseline['avg_net_recv_kb'], color='tab:blue',   linestyle='--', linewidth=1.5, label=f"Baseline Recv ({baseline['avg_net_recv_kb']:.1f} KB/s)")
#    axes[1].axhline(baseline['avg_net_send_kb'], color='tab:orange', linestyle='--', linewidth=1.5, label=f"Baseline Send ({baseline['avg_net_send_kb']:.1f} KB/s)")
#    axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Run-to-Run Consistency

In [ ]:
# Line plot showing variation across runs
plt.figure(figsize=(12, 6))
for workflow in df['workflow_file'].unique():
    workflow_data = df[df['workflow_file'] == workflow]
    plt.plot(workflow_data['run_number'], workflow_data['execution_time_seconds'], 
             marker='o', label=workflow)

plt.title('Execution Time Across Runs')
plt.xlabel('Run Number')
plt.ylabel('Execution Time (seconds)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 6. Correlation Analysis

In [ ]:
metrics = ['execution_time_seconds', 'avg_cpu_percent', 'peak_memory_mb', 
           'avg_disk_read_kb', 'avg_disk_write_kb']
correlation = df[metrics].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix of Performance Metrics')
plt.tight_layout()
plt.show()

print("\nKey Correlations:")
print(correlation['execution_time_seconds'].sort_values(ascending=False))

## 7. Performance Comparison Table

In [ ]:
comparison = df.groupby('workflow_file').agg({
    'execution_time_seconds': ['mean', 'std', 'min', 'max'],
    'avg_cpu_percent': 'mean',
    'peak_memory_mb': 'max',
    'avg_disk_read_kb': 'mean',
    'avg_disk_write_kb': 'mean'
}).round(2)

print("\nPerformance Comparison:")
print(comparison)

comparison.to_csv('results/performance_summary.csv')
print("\nSummary exported to: results/performance_summary.csv")

## 8. Tool Comparison: Argus vs zizmor vs ades

In [ ]:
# Compare execution times between tools
comparison_summary = df.groupby('tool')[['execution_time_seconds', 'avg_cpu_percent', 'peak_memory_mb', 'avg_disk_read_kb', 'avg_disk_write_kb']].agg(['mean', 'median', 'std']).round(3)
print('Performance Summary by Tool:')
print(comparison_summary)
print()

# Calculate speedup/slowdown ratio
argus_mean = df[df['tool'] == 'argus']['execution_time_seconds'].mean()
zizmor_mean = df[df['tool'] == 'zizmor']['execution_time_seconds'].mean()
ades_mean = df[df['tool'] == 'ades']['execution_time_seconds'].mean()
for other, other_mean in [('zizmor', zizmor_mean), ('ades', ades_mean)]:
    ratio = other_mean / argus_mean
    print(f'Execution Time Ratio ({other}/argus): {ratio:.2f}x')
    if ratio < 1:
        print(f'  → {other} is {1/ratio:.2f}x faster than Argus')
    else:
        print(f'  → Argus is {ratio:.2f}x faster than {other}')

### Execution Time Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Side-by-side box plot
sns.boxplot(data=df, x='tool', y='execution_time_seconds', ax=axes[0])
axes[0].set_title('Execution Time Distribution by Tool')
axes[0].set_xlabel('Tool')
axes[0].set_ylabel('Execution Time (seconds)')

# Grouped bar chart by workflow
workflow_means = df.groupby(['workflow_file', 'tool'])['execution_time_seconds'].mean().unstack()
workflow_means.plot(kind='bar', ax=axes[1], width=0.8)
axes[1].set_title('Average Execution Time by Workflow and Tool')
axes[1].set_xlabel('Workflow')
axes[1].set_ylabel('Execution Time (seconds)')
axes[1].legend(title='Tool')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('results/perf_exec_time.png', dpi=150, bbox_inches='tight')
plt.show()

### Resource Usage Comparison

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# CPU Usage
sns.boxplot(data=df, x='tool', y='avg_cpu_percent', ax=axes[0, 0])
axes[0, 0].set_title('CPU Usage Comparison')
axes[0, 0].set_xlabel('Tool')
axes[0, 0].set_ylabel('Average CPU (%)')

# Memory Usage
sns.boxplot(data=df, x='tool', y='peak_memory_mb', ax=axes[0, 1])
axes[0, 1].set_title('Memory Usage Comparison')
axes[0, 1].set_xlabel('Tool')
axes[0, 1].set_ylabel('Peak Memory (MB)')

# Disk Read
sns.boxplot(data=df, x='tool', y='avg_disk_read_kb', ax=axes[1, 0])
axes[1, 0].set_title('Disk Read Comparison')
axes[1, 0].set_xlabel('Tool')
axes[1, 0].set_ylabel('Average Disk Read (KB/s)')

# Disk Write
sns.boxplot(data=df, x='tool', y='avg_disk_write_kb', ax=axes[1, 1])
axes[1, 1].set_title('Disk Write Comparison')
axes[1, 1].set_xlabel('Tool')
axes[1, 1].set_ylabel('Average Disk Write (KB/s)')

plt.tight_layout()
plt.savefig('results/perf_resource_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

### Performance Ratio Analysis

In [ ]:
# Per-workflow execution time comparison across the three tools
workflow_means = df.groupby(['workflow_file', 'tool'])['execution_time_seconds'].mean().unstack()
argus_times = workflow_means['argus']
zizmor_times = workflow_means['zizmor']
ades_times = workflow_means['ades']
performance_ratio_z = (zizmor_times / argus_times).sort_values()
performance_ratio_a = (ades_times / argus_times).loc[performance_ratio_z.index]

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

axes[0].barh(performance_ratio_z.index.astype(str), performance_ratio_z.values,
             color=['green' if x < 1 else 'red' for x in performance_ratio_z], label='zizmor / argus')
axes[0].barh(performance_ratio_a.index.astype(str), performance_ratio_a.values,
             color=['#9b59b6' if x < 1 else '#e67e22' for x in performance_ratio_a], alpha=0.7, label='ades / argus')
axes[0].axvline(x=1, color='black', linestyle='--', linewidth=2, label='Equal performance')
axes[0].set_xscale('log')
axes[0].set_title('Performance Ratio by Workflow (log scale, < 1 means faster than Argus)')
axes[0].set_xlabel('Ratio (log scale)')
axes[0].set_ylabel('Workflow')
axes[0].legend()

workflow_means_sorted = workflow_means.loc[performance_ratio_z.index]
workflow_means_sorted.plot(kind='barh', ax=axes[1], width=0.8)
axes[1].set_xscale('log')
axes[1].set_title('Mean Execution Time by Workflow and Tool (log scale)')
axes[1].set_xlabel('Execution Time (seconds, log scale)')
axes[1].set_ylabel('Workflow')
axes[1].legend(title='Tool')

plt.tight_layout()
plt.savefig('results/perf_ratio.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nWorkflows where zizmor is faster than Argus: {(performance_ratio_z < 1).sum()}/{len(performance_ratio_z)}')
print(f'Workflows where ades is faster than Argus: {(performance_ratio_a < 1).sum()}/{len(performance_ratio_a)}')
print(f'\nZizmor/Argus average ratio: {performance_ratio_z.mean():.2f}')
print(f'Zizmor/Argus median ratio: {performance_ratio_z.median():.2f}')
print(f'Ades/Argus average ratio: {performance_ratio_a.mean():.2f}')
print(f'Ades/Argus median ratio: {performance_ratio_a.median():.2f}')

### Detailed Comparison Table

In [ ]:
# Create detailed comparison table
comparison_table = df.groupby(['workflow_file', 'tool']).agg({
    'execution_time_seconds': ['mean', 'std', 'min', 'max'],
    'avg_cpu_percent': 'mean',
    'peak_memory_mb': 'mean'
}).round(3)

# Flatten column names
comparison_table.columns = ['_'.join(col).strip() for col in comparison_table.columns]
comparison_table = comparison_table.reset_index()

print('Detailed Tool Comparison:')
display(comparison_table)

# Export comparison
comparison_table.to_csv('results/tool_comparison.csv', index=False)
print('\nComparison exported to: results/tool_comparison.csv')